# Example: Bandit Sigma-Learning

In this example, we apply the **epsilon-greedy multi-armed bandit** to learn the best CES elasticity $\sigma$ per sentiment regime. Instead of using the Session 2 heuristic $\sigma(\lambda_t) = \sigma_{\min} + (\sigma_{\max} - \sigma_{\min})/(1 + |\lambda_t|)$, we let the bandit discover which $\sigma$ value produces the highest utility in bearish, neutral, and bullish conditions.

> **By the end of this example, you will be able to:**
> * __Train the sigma-bandit on a single path:__ Run the epsilon-greedy bandit over a discrete sigma grid with separate instances per regime bin. Visualize reward convergence, exploration decay, and per-regime arm means.
> * __Backtest bandit-learned sigma across Monte Carlo paths:__ Compare three CES strategies (bandit-learned sigma, heuristic sigma, fixed sigma) on the same ensemble from Example 1. Evaluate distributional performance.
> * __Compare learned vs. heuristic sigma head-to-head:__ Produce a scorecard and paired-excess analysis showing where the bandit outperforms and where it underperforms the heuristic.

Let's see if a learner can beat a heuristic.

___

## Setup, Data and Prerequisites

In [ ]:
include("Include.jl");

### Constants


In [ ]:
# Sigma-bandit configuration
B₀ = 10_000.0
Δt = 1.0 / 252.0
L_short = 21
L_long = 63
L_growth = 10
GAIN = 10.0
offset = L_short + L_long
SCENARIO_SEED = 2026
SIGMA_GRID = [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]
BANDIT_ITERS_SINGLE = 500
BANDIT_ALPHA = 0.1
LAMBDA_THRESHOLD = 0.5
TRIGGER_MAX_DRAWDOWN = 0.15
TRIGGER_MAX_TURNOVER = 0.50
SIGMA_BOUNDS = (0.5, 5.0)
ALLOCATION_EPSILON = 0.1


Load the same Session 1 artifacts and regenerate the same 500-path scenario used in Example 1 (EWLS Engine Replay). The deterministic seed ensures identical paths.

In [ ]:
my_tickers, sim_estimates, g_f, sim_params, market_model, portfolio, calib, start_prices, N, scenario = let
    # --- Step 1: Load S1 artifacts ---
    minvar = load_results(joinpath(_PATH_TO_DATA_S1, "minvar-allocation.jld2"));
    my_tickers    = minvar["my_tickers"]::Vector{String};
    sim_estimates = minvar["sim_estimates"];
    g_f           = haskey(minvar, "g_f") ? Float64(minvar["g_f"]) : Float64(minvar["r_f"]);
    sim_params = Dict{String,Tuple{Float64,Float64,Float64}}(
        e.ticker => (e.\u03b1, e.\u03b2, e.\u03c3_\u03b5) for e in sim_estimates
    );

    # --- Step 2: Surrogates and scenario ---
    market_model = MyMarketSurrogateModel();
    portfolio    = MyPortfolioSurrogateModel();
    calib        = load_results(joinpath(_PATH_TO_DATA_S1, "sim-parameter-estimates.jld2"));
    snap        = MyCurrentPrices();
    snap_lookup = Dict(snap["tickers"] .=> snap["prices"]);
    start_prices = Dict(t => snap_lookup[t] for t in my_tickers);

    # --- Step 3: Dimensions ---
    N         = length(my_tickers);
    n_trading_days   = 252;
    T_total          = offset + n_trading_days;

    # --- Step 4: Regenerate scenario (same seed as Example 1) ---
    Random.seed!(2026);
    scenario = generate_hybrid_scenario(
        market_model, portfolio, calib, my_tickers;
        n_paths = 500, n_steps = T_total,
        start_prices = start_prices, label = "S3 Sigma-Bandit", seed = 2026);

    println("Loaded $(N) tickers, generated $(scenario.n_paths)-path scenario")
    my_tickers, sim_estimates, g_f, sim_params, market_model, portfolio, calib, start_prices, N, scenario
end


___
## Task 1: Train the Sigma-Bandit on a Single Path
We extract the first path from the scenario, build the rebalancing context, and run the sigma-bandit with 6 arms ($\sigma \in \{0.5, 1.0, 1.5, 2.0, 3.0, 5.0\}$) and a regime threshold $\theta = 0.5$. The bandit learns independently in bearish, neutral, and bullish bins.

> __What should you see?__
>
> The reward trace should show initial exploration (noisy) converging to a stable level as the bandit discovers the best arm per regime. The per-regime bar chart should reveal whether different regimes prefer different sigma values.

In [ ]:
bandit_result = let
    # --- Step 1: Extract first path ---
    p_idx = 1;
    T_total = scenario.n_steps;
    mkt = scenario.market_paths[p_idx, :];
    pmatrix = zeros(T_total, N + 1);
    pmatrix[:, 1] = 1:T_total;
    for k in 1:N
        pmatrix[:, k + 1] = scenario.price_paths[p_idx, :, k];
    end

    # --- Step 2: Compute lambda series ---
    ema_s = compute_ema(mkt; window = L_short);
    ema_l = compute_ema(mkt; window = L_long);
    lambda_series = compute_lambda(ema_s, ema_l; G = GAIN);
    lambda_series[1:offset] .= 0.0;
    gm_raw = compute_market_growth(mkt; \u0394t = \u0394t);
    gm_ema = compute_ema(gm_raw; window = L_growth);

    # --- Step 3: Build context ---
    ctx = build(MyRebalancingContextModel, (
        B = B\u2080, tickers = my_tickers, marketdata = pmatrix,
        marketfactor = gm_ema, sim_parameters = sim_params,
        lambda = 0.0, \u0394t = \u0394t, epsilon = ALLOCATION_EPSILON
    ));

    # --- Step 4: Build and run sigma-bandit ---
    \u03c3_grid = SIGMA_GRID;
    bandit = build(MySigmaBanditModel, (
        sigma_grid = \u03c3_grid, n_iterations = BANDIT_ITERS_SINGLE, alpha = BANDIT_ALPHA, lambda_threshold = LAMBDA_THRESHOLD
    ));

    time_indices = collect((offset + 1):T_total);
    bandit_result = solve_sigma_bandit(bandit, ctx, lambda_series, time_indices);

    # --- Step 5: Report best sigma per regime ---
    println("Sigma-Bandit Results (Path $(p_idx)):")
    for regime in [:bearish, :neutral, :bullish]
        \u03c3_best = bandit_result.best_sigma_per_regime[regime];
        counts = bandit_result.arm_counts_per_regime[regime];
        means = bandit_result.arm_means_per_regime[regime];
        println("  $(regime): best \u03c3 = $(\u03c3_best), pulls = $(sum(counts)), arm means = $(round.(means, digits=3))")
    end

    # --- Step 6: Visualize ---
    p1 = plot(bandit_result.reward_history, label="Reward", alpha=0.3, color=:gray50,
        ylabel="CES Utility", title="Sigma-Bandit: Reward Trace", size=(800, 250))

    p2 = plot(bandit_result.exploration_history, label="\u03b5_t", color=:coral, linewidth=2,
        ylabel="\u03b5", title="Exploration Decay", xlabel="Iteration", size=(800, 200))

    display(plot(p1, p2, layout=grid(2,1, heights=[0.55, 0.45]), size=(800, 450)))

    # Per-regime arm means bar chart
    p3 = plot(layout=(1,3), size=(900, 300), title=["Bearish" "Neutral" "Bullish"])
    for (j, regime) in enumerate([:bearish, :neutral, :bullish])
        bar!(p3[j], string.(\u03c3_grid), bandit_result.arm_means_per_regime[regime],
            label="", color=:steelblue, xlabel="\u03c3", ylabel=j==1 ? "Mean Reward" : "")
    end
    display(p3)
    bandit_result
end


___
## Task 2: Backtest Bandit-Learned Sigma Across All Paths
We take the sigma map learned on path 1 and apply it across all 500 paths via `backtest_sigma_bandit`. We also run the heuristic $\sigma(\lambda)$ engine and a fixed $\sigma = 2.0$ engine as baselines.

> __What should you see?__
>
> The bandit-learned sigma should match or beat the heuristic on median metrics. The fixed sigma baseline should underperform both adaptive approaches, confirming that regime-awareness matters.

In [ ]:
bandit_bt = let
    # --- Step 1: Run bandit-learned sigma engine ---
    sigma_map = bandit_result.best_sigma_per_regime;
    bandit_bt = backtest_sigma_bandit(scenario, my_tickers, sim_params, sigma_map;
        B\u2080 = B\u2080, offset = offset, L_short = L_short, L_long = L_long,
        GAIN = GAIN, L_growth = L_growth, lambda_threshold = 0.5);

    # --- Step 2: Run heuristic sigma engine (adaptive_sigma=true) ---
    n_trading = scenario.n_steps - offset;
    heur_final_wealth = zeros(scenario.n_paths);
    heur_max_dd = zeros(scenario.n_paths);
    heur_sharpe = zeros(scenario.n_paths);

    for p in 1:scenario.n_paths
        mkt = scenario.market_paths[p, :];
        ema_s = compute_ema(mkt; window = L_short);
        ema_l = compute_ema(mkt; window = L_long);
        \u03bb_p = compute_lambda(ema_s, ema_l; G = GAIN);
        \u03bb_p[1:offset] .= 0.0;
        gm_raw = compute_market_growth(mkt; \u0394t = \u0394t);
        gm_e = compute_ema(gm_raw; window = L_growth);

        pmatrix = zeros(scenario.n_steps, N + 1);
        pmatrix[:, 1] = 1:scenario.n_steps;
        for k in 1:N
            pmatrix[:, k + 1] = scenario.price_paths[p, :, k];
        end

        ctx = build(MyRebalancingContextModel, (
            B = B\u2080, tickers = my_tickers, marketdata = pmatrix,
            marketfactor = gm_e, sim_parameters = sim_params,
            lambda = 0.0, \u0394t = \u0394t, epsilon = 0.1
        ));
        rules = build(MyTriggerRules, (
            max_drawdown = 0.15, max_turnover = 0.50,
            rebalance_schedule = ones(Int, n_trading)
        ));
        results = run_rebalancing_engine(ctx, rules, \u03bb_p;
            offset = offset, allocator = :ces, adaptive_sigma = true, sigma_bounds = (0.5, 5.0));
        w = compute_wealth_series(results, pmatrix, my_tickers; offset = offset);

        heur_final_wealth[p] = w[end];
        ret = diff(w) ./ w[1:end-1];
        pk = accumulate(max, w);
        heur_max_dd[p] = maximum((pk .- w) ./ pk);
        vol = std(ret) * sqrt(252);
        heur_sharpe[p] = vol > 0 ? (w[end]/w[1] - 1.0) / vol : 0.0;
    end

    # --- Step 3: Load frozen baseline from Example 1 ---
    ewls_data = load_results(joinpath(_PATH_TO_DATA, "ewls-replay-results.jld2"));
    frozen_fw = ewls_data["frozen_final_wealth"];

    # --- Step 4: Scorecard ---
    println("="^70)
    println("  CES SIGMA STRATEGIES: Distributional Comparison ($(scenario.n_paths) paths)")
    println("="^70)
    println("  Metric               Bandit-\u03c3      Heuristic-\u03c3    Fixed \u03c3=2.0")
    println("  " * "-"^55)
    println("  Median W/W\u2080          $(round(median(bandit_bt.final_wealth)/B\u2080, digits=3))           $(round(median(heur_final_wealth)/B\u2080, digits=3))           $(round(median(frozen_fw)/B\u2080, digits=3))")
    println("  Median Max DD        $(round(median(bandit_bt.max_drawdowns)*100, digits=1))%           $(round(median(heur_max_dd)*100, digits=1))%           --")
    println("  Median Sharpe        $(round(median(bandit_bt.sharpe_ratios), digits=3))           $(round(median(heur_sharpe), digits=3))           --")

    # --- Step 5: Save for Example 3 ---
    save_results(joinpath(_PATH_TO_DATA, "sigma-bandit-results.jld2"), Dict(
        "bandit_final_wealth" => bandit_bt.final_wealth,
        "bandit_max_dd" => bandit_bt.max_drawdowns,
        "bandit_sharpe" => bandit_bt.sharpe_ratios,
        "heur_final_wealth" => heur_final_wealth,
        "heur_max_dd" => heur_max_dd,
        "heur_sharpe" => heur_sharpe,
        "best_sigma_per_regime" => bandit_result.best_sigma_per_regime,
    ));
    println("\nSaved to sigma-bandit-results.jld2")
    bandit_bt
end


___
## Task 3: Learned Sigma vs. Heuristic Sigma: Head-to-Head
We compute the per-path excess wealth (bandit minus heuristic) and visualize the distribution. This paired comparison controls for path-level variation and isolates the effect of the sigma selection method.

> __What should you see?__
>
> If the bandit discovered genuinely better sigma values, the excess distribution should be right-shifted (positive median). If the heuristic was already near-optimal, the distribution should be centered near zero.

In [ ]:
let
    # --- Step 1: Load bandit and heuristic results ---
    data = load_results(joinpath(_PATH_TO_DATA, "sigma-bandit-results.jld2"));
    bandit_fw = data["bandit_final_wealth"];
    heur_fw   = data["heur_final_wealth"];

    # --- Step 2: Paired excess ---
    excess = bandit_fw .- heur_fw;
    win_rate = mean(excess .> 0);

    println("Paired Excess (Bandit - Heuristic):")
    println("  Win rate:      $(round(win_rate*100, digits=1))%")
    println("  Median excess: \$$(round(median(excess), digits=0))  ($(round(median(excess)/B\u2080*100, digits=2))% of W\u2080)")
    println("  Mean excess:   \$$(round(mean(excess), digits=0))  ($(round(mean(excess)/B\u2080*100, digits=2))% of W\u2080)")

    # --- Step 3: Histogram ---
    p = histogram(excess ./ B\u2080, bins = 60, alpha = 0.7, color = :steelblue,
        xlabel = "Per-Path Excess (W_bandit - W_heuristic) / W\u2080",
        ylabel = "Count", title = "Bandit-\u03c3 vs Heuristic-\u03c3: Paired Excess",
        legend = false, size = (800, 400))
    vline!(p, [0.0], lw = 2, c = :black, linestyle = :dash)
    vline!(p, [median(excess) / B\u2080], lw = 2, c = :coral, label = "")
    display(p)

    # --- Step 4: Report learned sigma map ---
    sigma_map = data["best_sigma_per_regime"];
    println("\nLearned sigma map:")
    for regime in [:bearish, :neutral, :bullish]
        println("  $(regime): \u03c3 = $(sigma_map[regime])")
    end
end

___
## Summary
This example trained an epsilon-greedy sigma-bandit on a single path, applied the learned sigma map across 500 Monte Carlo paths, and compared the results head-to-head against the Session 2 heuristic. The learned sigma map and backtest results are saved to `sigma-bandit-results.jld2` for the validation report in Example 3.

### Key Takeaways
* __The bandit discovers regime-specific sigma values:__ Different sentiment regimes may prefer different elasticity levels. The bandit can break the symmetry that the heuristic imposes via $|\lambda|$.
* __Per-regime learning isolates the effect of sigma choice:__ By holding the preference weights and trigger rules fixed, the backtest comparison isolates how much the sigma selection method contributes to performance.
* __The heuristic is a strong baseline:__ If the excess distribution is centered near zero, the hand-designed formula was already close to optimal on this data, and the bandit's value lies in confirming that rather than discovering something new.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.